In [ ]:
print("hello world")

# 安装unsloth
一定要安装官方notebook版本安装：https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-Alpaca.ipynb

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

# 处理数据
将VideOCR识别的srt文件转换为jsonl格式

In [ ]:
import re
import json

def process_mixed_srt(file_path):
    """
    专门处理中英双语混排的 SRT 文件
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # 1. 按照 SRT 序号切分块
    # 匹配模式：数字序号 + 时间轴
    blocks = re.split(r'\n\d+\n\d{2}:\d{2}:\d{2},\d{3} --> \d{2}:\d{2}:\d{2},\d{3}\n', content)

    refined_data = []

    for block in blocks:
        lines = [l.strip() for l in block.split('\n') if l.strip()]

        # 逻辑：如果一个块里有两行，通常第一行是中文，第二行是英文
        # 或者使用正则表达式判断中英
        if len(lines) >= 2:
            # 简单的正则：判断哪行包含中文
            zh_line = ""
            en_line = ""
            for l in lines:
                if re.search(r'[\u4e00-\u9fa5]', l): # 包含汉字
                    zh_line = l
                else:
                    en_line = l

            if zh_line and en_line:
                refined_data.append({"en": en_line, "zh": zh_line})

        # 如果只有一行（比如第1个块的 Joyner），在这个场景下通常是名字或语气词，建议过滤或跳过
        elif len(lines) == 1:
            # 如果你想保留单行作为背景，可以取消下面注释
            # refined_data.append({"en": lines[0], "zh": lines[0]})
            pass

    return refined_data

def convert_to_sharegpt_custom(pairs, output_file, window_size=5):
    """
    将提取的对齐数据转换为 ShareGPT 格式
    """
    dataset = []
    system_prompt = "你是一位精通美国嘻哈文化和俚语的翻译专家。你的任务是翻译歌词，并对其中的文化梗进行深度解析。"

    for i in range(0, len(pairs), window_size):
        chunk = pairs[i : i + window_size]
        en_text = "\n".join([p["en"] for p in chunk])
        zh_text = "\n".join([p["zh"] for p in chunk])

        sample = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"歌词原文：\n{en_text}"},
                {
                    "role": "assistant",
                    "content": f"歌词译文：\n{zh_text}"
                }
            ]
        }
        dataset.append(sample)

    with open(output_file, 'w', encoding='utf-8') as f:
        for entry in dataset:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

    print(f"处理完成！生成了 {len(dataset)} 个训练单元。")

# --- 执行 ---
# 假设你的文件叫 lyrics.srt
# pairs = process_mixed_srt("lyrics.srt")
# convert_to_sharegpt_custom(pairs, "hiphop_expert_train.jsonl")

# -- 批量处理文件夹下的所有 SRT 文件 ---
import os
import glob

def batch_process_srt(input_folder, output_jsonl, window_size=5):
    """
    遍历文件夹下的所有 srt 文件并合并转换
    """
    # 1. 获取所有 srt 文件路径
    # 使用 os.path.join 确保路径在 Linux 环境下正确
    search_path = os.path.join(input_folder, "*.srt")
    srt_files = glob.glob(search_path)
    
    if not srt_files:
        print(f"❌ 在文件夹 {input_folder} 中没有找到任何 .srt 文件")
        return

    print(f"📂 找到 {len(srt_files)} 个文件，准备开始处理...")

    all_pairs = []
    
    # 2. 逐个处理文件并汇总 pairs
    for file_path in srt_files:
        try:
            print(f"正在解析: {os.path.basename(file_path)}")
            file_pairs = process_mixed_srt(file_path)
            all_pairs.extend(file_pairs)
        except Exception as e:
            print(f"⚠️ 处理文件 {file_path} 时出错: {e}")

    # 3. 统一转换为 ShareGPT 格式
    if all_pairs:
        convert_to_sharegpt_custom(all_pairs, output_jsonl, window_size)
        print(f"✅ 批量处理成功！总计歌词对: {len(all_pairs)}")
        print(f"📦 最终训练文件已保存至: {output_jsonl}")
    else:
        print("分拣后的有效歌词对为空，请检查 SRT 格式。")

# --- 执行区 ---
# 请确保你在 VS Code 侧边栏创建了这个文件夹并上传了文件
input_dir = "/content/my_lyrics"  # 你的 srt 存放目录
output_name = "hiphop_finetune_data.jsonl"

# 如果文件夹不存在，先创建它（提醒你手动上传文件）
if not os.path.exists(input_dir):
    os.makedirs(input_dir)
    print(f"已创建目录 {input_dir}，请将所有 SRT 文件拖入该目录后重新运行。")
else:
    batch_process_srt(input_dir, output_name, window_size=6)

In [ ]:
from datasets import load_dataset

# 加载你的 JSONL 文件
dataset = load_dataset("json", data_files="/content/train_data/hiphop_finetune_data.jsonl", split="train")

# 检查一下数据格式
print(f"✅ 数据加载成功，共 {len(dataset)} 条样本")
print(f"样例数据: {dataset[0]['messages'][1]['content'][:50]}...")

In [ ]:
from unsloth import FastLanguageModel
import torch

# 1. 加载 Qwen2.5-7B-Instruct
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

# 2. 同样的推理测试（看看没训练前 Qwen 的实力）
FastLanguageModel.for_inference(model)

test_lyric = "Life can bring much pain. There are many variations of pain and I'm a connoisseur of all."
messages = [
    {"role": "system", "content": "你是一位精通嘻哈文化和 J. Cole 风格的翻译专家。"},
    {"role": "user", "content": f"翻译并解析歌词: '{test_lyric}'"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        input_ids = inputs,
        max_new_tokens = 512,
        temperature = 0.7,
        do_sample = True,
        pad_token_id = tokenizer.pad_token_id
    )

print("--- [Qwen2.5 原始翻译结果] ---")
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

In [ ]:
from unsloth import FastLanguageModel
from unsloth import standardize_sharegpt
from trl import SFTTrainer
from transformers import TrainingArguments

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,               # LoRA 的秩，值越大拟合能力越强，16 是平衡点
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",], # 针对 Qwen 的全量层优化
    lora_alpha = 16,
    lora_dropout = 0,     # 设置为 0 性能最强
    bias = "none",        # 设置为 none 性能最强
    use_gradient_checkpointing = "unsloth", # 节省显存，T4 必备
    random_state = 3407,
    use_rslora = False,   # 我们先用标准的 LoRA
    loftq_config = None, 
)

print("✅ LoRA 适配器已挂载，现在可以开始微调了！")

# 然后再运行之前的 trainer = SFTTrainer(...) 定义部分

# 1. 准备数据（Qwen 也支持 standardize_sharegpt）
dataset = standardize_sharegpt(dataset)

def formatting_prompts_func(examples):
    instructions = examples["messages"]
    # 使用 Qwen 的模板
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in instructions]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched=True)

# 2. 训练配置（既然基座很强，我们可以用较小的学习率）
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        max_steps = 60,         # Qwen 学得快，60步先看看效果
        learning_rate = 5e-5,   # 稍微调低，保护它原有的中文智商
        fp16 = True,
        optim = "adamw_8bit",
        logging_steps = 1,
        output_dir = "qwen_jcole_outputs",
    ),
)

trainer.train()

In [ ]:
FastLanguageModel.for_inference(model)

# 2. 准备 Prompt
test_quote = "No such thing as a life that's better than yours."
messages = [
    {"role": "user", "content": f"翻译 J. Cole 在《Love Yourz》里的这句名言并深度解析：'{test_quote}'"}
]

# 3. 编码输入并手动生成 Attention Mask
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# 4. 显式设置生成参数，消除警告
outputs = model.generate(
    input_ids = inputs,
    attention_mask = (inputs != tokenizer.pad_token_id).long(), # 显式设置 Mask
    max_new_tokens = 512,
    temperature = 0.5,      # 稍微降低随机性，让解析更深沉
    repetition_penalty = 1.2,
    do_sample = True,
    pad_token_id = tokenizer.eos_token_id
)

print("--- [微调后的 J. Cole 专家模型最终成果] ---")
final_output = tokenizer.decode(outputs[0], skip_special_tokens = True)
# 过滤掉 Prompt 部分，只看 Assistant 的回答
print(final_output.split("assistant")[-1].strip())